# Import repaired BAFU EcoSpold files

Create or reuse a local Brightway project with **ecoinvent biosphere 3.10**, then try importing the repaired EcoSpold 1 files. Select the **bw** kernel and run through the migration and inspection cells. The optional database-write section stops while exchanges remain unresolved.

The repaired files must already exist. To generate them, run this from the repository root:

```bash
conda run --no-capture-output -n bw python "scripts/ecospold importer/repair_all.py"
```

Project storage is `artifacts/brightway/` (ignored by Git). The first project setup downloads Brightway's biosphere archive and requires internet access.

In [ ]:
import os
import sys
from pathlib import Path


ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts/ecospold importer").is_dir()
)
SOURCE = ROOT / "data/processed/ecospold1-schema-fixed"
STORAGE = ROOT / "artifacts/brightway"

PROJECT = "bafu-2026-biosphere-310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
DATABASE = "BAFU:2026"

# Configure storage and the local helper path before importing Brightway.
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["BRIGHTWAY2_DIR"] = str(STORAGE)
sys.path.insert(0, str(ROOT / "scripts/ecospold importer"))

In [ ]:
# Run the setup cell above first.
import bw2data as bd
import bw2io as bi
from date_compat import xml_date_parser
from timestamp_compat import iso_timestamp_parser

## Create the project

Reuse the same project as the Python import script, or create it from Brightway's biosphere 3.10 archive on first use.

In [ ]:
if PROJECT not in bd.projects:
    bi.install_project("ecoinvent-3.10-biosphere", project_name=PROJECT)
    
bd.projects.set_current(PROJECT)

In [ ]:
bd.databases

## Extract the repaired files

Use the local date/timestamp adapters and the standard EcoSpold 1 importer strategies. The XML files remain unchanged. The `importer` object stays available for inspection.

In [ ]:
with iso_timestamp_parser(), xml_date_parser():
    importer = bi.SingleOutputEcospold1Importer(str(SOURCE), DATABASE, use_mp=False)

importer.apply_strategies()

## Apply the technosphere migrations

Edit the [general mappings](../schemas/mappings/bafu-2026-technosphere.json) or the [source-file-specific mappings](../schemas/mappings/bafu-2026-technosphere-context.json). The helper reads both files on every run, registers them in the current project, applies the corrections, and reruns technosphere linking.

The second file distinguishes identical exchange labels used by different consuming datasets. Its source-file marker is temporary and is removed after migration. See the [mapping notes](../schemas/mappings/README.md) for evidence and documented assumptions.

After changing either JSON file, rerun the extraction cell above and the following cells. Raw and repaired XML files remain unchanged.

In [ ]:
from technosphere_migrations import apply_technosphere_migrations

apply_technosphere_migrations(importer, ROOT / "schemas/mappings")

In [ ]:
importer.match_database(
    fields=["name", "reference product", "location"], edge_kinds=["technosphere"]
)

## Normalize biosphere categories and link

Apply the [category migration](../schemas/mappings/bafu-2026-biosphere-categories.json) before matching against biosphere 3.10. This translates BAFU compartment labels while keeping flow names, units, amounts, uncertainty, and comments unchanged. Matching uses the complete **name, categories, and unit**.

Unresolved flows remain in `importer.data`. The helper writes their occurrence counts, example source files, and diagnostic groups to `reports/generated/biosphere-unlinked-after-categories.json`. See the [biosphere migration notes](../docs/bafu-2026-biosphere-migrations.md). Rerun extraction and all migration cells after editing the mappings.

Category normalization is the first pass; the remaining name, unit, and compartment mismatches still require review before a complete inventory can be written.

In [ ]:
from biosphere_migrations import apply_biosphere_category_migration

biosphere_result = apply_biosphere_category_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-categories.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-categories.json",
)
biosphere_result

## Apply reviewed biosphere names and units

The [flow migration](../schemas/mappings/bafu-2026-biosphere-flows.json) handles explicit chemical/name aliases, equivalent particulate and biogenic labels, redundant `/m3` suffixes, and exact Bq → kBq or kWh → MJ conversions. Compartments stay unchanged. bw2io applies the documented conversion factors to the amounts and their uncertainty parameters.

Every rule must match exactly one target flow in biosphere 3.10. This stage's unresolved-flow report is `reports/generated/biosphere-unlinked-after-flows.json`; the category-only result is saved separately. See the [evidence and limitations](../docs/bafu-2026-biosphere-flow-migrations.md). Rerun extraction and all migration cells after editing a mapping.

In [ ]:
from biosphere_migrations import apply_biosphere_flow_migration

biosphere_flow_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-flows.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-flows.json",
)
biosphere_flow_result

## Apply historical biosphere catalog mappings

Use the [catalog migration](../schemas/mappings/bafu-2026-biosphere-catalog.json) to follow documented ecoinvent UUID correspondences, catalog synonyms, and historical renames. Source CAS values are included in matching, so an explicitly identified chromium(VI) exchange is handled separately from the generic Chromium label. Amounts, units, compartments, and original CAS metadata are retained.

The [catalog evidence notes](../docs/bafu-2026-biosphere-catalog-migrations.md) describe the source catalogs and confidence limits.

In [ ]:
biosphere_catalog_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-catalog.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-catalog.json",
)
biosphere_catalog_result

## Map documented resource names and water units

Apply the [resource migration](../schemas/mappings/bafu-2026-biosphere-resources.json). Legacy ore-composition names are replaced by element names according to ecoinvent's documented 3.10 migration. Water emissions expressed in kilograms are converted to the target catalog's cubic-metre convention (1,000 kg per m³), including their uncertainty parameters. Raw XML remains unchanged.

See the [resource evidence notes](../docs/bafu-2026-biosphere-resource-migrations.md). This step retains every exchange and writes `reports/generated/biosphere-unlinked-after-resources.json`.

In [ ]:
biosphere_resource_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-resources.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-resources.json",
)
biosphere_resource_result

## Apply additional reviewed biosphere mappings

Apply the [reviewed migration](../schemas/mappings/bafu-2026-biosphere-reviewed.json): chemical spelling/synonym corrections, omitted resource subcategories, and the approved regional-water and land-class mappings. Regional emissions become generic `Water`; regional withdrawals retain their water type (river, lake, cooling, or unspecified origin), with the original name, unit, categories, and region retained in each exchange's `bafu original biosphere` metadata. Generic LCIA factors do not use that retained region automatically. Water kg → m³ uses the documented factor 0.001, including uncertainty rescaling.

The approved broader industrial-area and intensive-forest classes also retain their original detailed labels in `bafu original biosphere` metadata. Their linked LCIA factors use the broader class. Unsupported flows remain unlinked. See the [evidence and limits](../docs/bafu-2026-biosphere-reviewed-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-reviewed.json`.

In [ ]:
biosphere_reviewed_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-reviewed.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-reviewed.json",
)

## Correct reviewed water and resource contexts

Apply the [water/context migration](../schemas/mappings/bafu-2026-biosphere-water-context.json). It resolves well-water and fossil-water resource names, corrects explicitly identified resource compartments, and covers the remaining supported regional water labels. Fossil-water emissions use the existing `water / fossil well` target. Process-water withdrawal labels retain their stated groundwater category where present; other unspecified-origin withdrawals use the general water-resource class.

All rules retain their original name, unit, and categories on the exchange, with the region where stated. Water kg → m³ uses the documented 0.001 factor. See the [evidence and scope](../docs/bafu-2026-biosphere-water-context-migrations.md).

In [ ]:
biosphere_water_context_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-water-context.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-water-context.json",
)

## Apply reviewed chemical names and elemental mass

Apply the [elemental-mass migration](../schemas/mappings/bafu-2026-biosphere-chemistry.json): TiO₂ → titanium, KCl → potassium, and barite (BaSO₄) → barium. The approved factors represent the contained element's mass, so both source and target units are kilograms while their material basis differs. bw2io rescales the amount and uncertainty; the helper checks the factor against the stated formula and atomic weights.

Each exchange retains its original compound label, unit, and categories in `bafu original biosphere`, plus the formula, target element, and atomic weights in `bafu elemental conversion`. See the [evidence and factors](../docs/bafu-2026-biosphere-chemistry-migrations.md).

Five further catalog/chemical aliases cover VOC of unspecified origin, dimethylformamide, DSMA, TCMTB, and Tin (II). Their full compartments, units, and amounts are retained; original names remain on the exchanges.

In [ ]:
biosphere_chemistry_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-chemistry.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-chemistry.json",
)

## Apply the approved resource-gas unit assumption

The [gas-unit migration](../schemas/mappings/bafu-2026-biosphere-gas-units.json) maps natural-gas and mine-gas resources from `cubic meter` or `normal cubic meter` to their unique `standard cubic meter` targets. This is the approved **1:1 label assumption**: source reference conditions remain unknown, and no temperature/pressure conversion is performed.

Original names, categories, and units remain in `bafu original biosphere`; `bafu unit assumption` records the decision. No bw2io multiplier is used, so amounts and all uncertainty fields stay exactly unchanged. See the [scope and verification](../docs/bafu-2026-biosphere-gas-unit-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-gas-units.json`.

In [ ]:
biosphere_gas_unit_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-gas-units.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-gas-units.json",
)
biosphere_gas_unit_result

## Apply documented land and resource mappings

The [land/resource migration](../schemas/mappings/bafu-2026-biosphere-land-resources.json) follows official historical renames for seabed infrastructure, non-irrigated crop classes, and graphite. It also applies the approved broader classes for tropical rainforest and organic cropland/pasture, corrects the peat resource compartment, completes three approved regional-water aliases, and links a helium resource whose source dataset explicitly describes extraction from natural gas.

All incoming labels remain in `bafu original biosphere`, including Europe for the water rows. Amounts, units, and uncertainty are unchanged. The broader land targets use generic LCIA factors; retained tropical/organic detail is not automatically characterized. See the [evidence and validation](../docs/bafu-2026-biosphere-land-resource-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-land-resources.json`.

In [ ]:
biosphere_land_resource_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-land-resources.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-land-resources.json",
)
biosphere_land_resource_result

## Apply the reviewed follow-up mappings

The [follow-up migration](../schemas/mappings/bafu-2026-biosphere-followup.json) applies the approved broader unspecified-land classes and generic Swiss rail-land targets. Original labels and categories remain in `bafu original biosphere`; rail rows also retain `region: CH`. Generic LCIA does not automatically use these retained distinctions.

It also corrects Cesium-136 to Caesium-136 with an exact Bq → kBq conversion (×0.001), classifies the reviewed limestone/petroleum CO₂ exchange as fossil, and links three exact 2-chlorophenol synonyms in water. Every other amount and uncertainty field stays unchanged. See the [evidence and limitations](../docs/bafu-2026-biosphere-followup-migrations.md). This stage writes `reports/generated/biosphere-unlinked-after-followup.json`.

In [ ]:
biosphere_followup_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-followup.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-followup.json",
)
biosphere_followup_result

## Apply reviewed compartment and aggregate mappings

The [compartment migration](../schemas/mappings/bafu-2026-biosphere-compartments.json) moves the two landfill-mass/carbon indicators from `natural resource / in ground` to their official `inventory indicator / waste` category. Their names, amounts, signs, units, and uncertainty stay unchanged; original categories are retained in `bafu original biosphere`.

These existing biosphere targets are ecological-scarcity inventory indicators.

The same stage applies the approved generic-air targets for 913 high-altitude emissions, retaining `air / lower stratosphere + upper troposphere` in audit metadata. Amounts and uncertainty stay unchanged; generic LCIA will no longer distinguish their altitude. It also applies the approved broader compartments for 14 urban-air/surface-water/industrial-soil emissions, 75 suspended-solids emissions to fossil water, and 38 indoor TCDD emissions. Five further surface-water exchanges use the same approved generic-water approach after checking their exact chemical aliases. Every original compartment remains in audit metadata. Generic factors do not automatically represent the retained indoor exposure or aquifer distinction.

See the [definitions and verification](../docs/bafu-2026-biosphere-compartment-migrations.md). The final unresolved report is `reports/generated/biosphere-unlinked.json`.

The same file applies the approved TOC → Organic carbon mappings in urban air and generic soil, retaining the TOC labels. These targets have no factors in the 668 installed LCIA method components checked. Seven generic-air TOC rows remain unresolved.

It also maps 200 `water / river, long-term` emissions to their reviewed surface-water targets, retaining the full original timing category in `bafu original biosphere`. **Standard LCIA loses this timing distinction and may characterize them in methods labelled `no LT`.** All amounts and uncertainty fields remain unchanged.

In [ ]:
biosphere_compartment_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-compartments.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-compartments.json",
)
biosphere_compartment_result

## Apply source corrections and approved broader resources

The [source-review migration](../schemas/mappings/bafu-2026-biosphere-source-review.json) corrects the single `Metam-sodium dihydrate` label whose original agricultural report gives **Metam-sodium** on an active-ingredient mass basis. Matching includes the source CAS number. Original labels stay in audit metadata; amounts and uncertainty remain unchanged. See the [source evidence](../docs/bafu-2026-biosphere-source-review.md).

The same file applies the approved `forest, natural` → `forest, unspecified` mapping and generic water withdrawals in their existing in-water/in-ground compartments. Original forest, surface/process-water, and groundwater-cooling labels are retained; cooling rows also retain `region: RER`. Four water rows convert kg → m³ using the established 1,000 kg/m³ convention, scaling uncertainty consistently. Standard LCIA uses the broader target definitions.

In [ ]:
biosphere_source_review_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-source-review.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-source-review.json",
)
biosphere_source_review_result

## Apply mappings for identified source files

The [source-specific migration](../schemas/mappings/bafu-2026-biosphere-context.json) applies the approved railway-land proxies to 14 sealed-soil exchanges in seven identified railway datasets. Seven sealed-soil exchanges in plastics remain unresolved. These proxies use general rail-network factors; the original sealing distinction remains in audit metadata.

One deionised-water dataset also receives the source-supported `Waste water/m3` → `Water` correction. Its report identifies the 0.00011 m³ flow as a same-basin water return. Other wastewater labels remain unresolved. See the [evidence and scope](../docs/bafu-2026-biosphere-context-migrations.md).

Rules match the exact source filename as well as type, name, complete categories, unit, and source CAS. The helper supplies a temporary filename field to bw2io and removes it afterwards. All original labels, amounts, uncertainty fields, comments, and source CAS values are retained.

In [ ]:
biosphere_context_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-context.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-context.json",
)
biosphere_context_result

## Apply source-verified plastics corrections

The [plastics source-review migration](../schemas/mappings/bafu-2026-biosphere-plastics-source-review.json) uses the original EPS inventory and its referenced flow definitions to identify calcium chloride as CaCl₂ and magnesium chloride as MgCl₂. It converts their mass to contained calcium or magnesium in the same in-ground compartment, using the established elemental-resource approach. A third rule restores the original chromium-VI identity of a row mislabelled P-cyanophenol. Its amount, uncertainty, and soil/industrial compartment stay unchanged. A fourth rule restores a mislabelled PAH row and uses the approved generic-soil target, retaining its original label and industrial-soil compartment in metadata. Its amount and uncertainty stay unchanged; generic LCIA loses the industrial-soil distinction. The rules apply only to the identified EPS source file.

Original labels and the formula/atomic-weight basis remain in exchange metadata. bw2io scales amounts and uncertainty together. Original BAFU CAS fields remain unchanged. See the [source evidence and conversion factors](../docs/bafu-2026-biosphere-plastics-source-review.md).

In [ ]:
biosphere_plastics_source_review_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-plastics-source-review.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-plastics-source-review.json",
)
biosphere_plastics_source_review_result

## Restore two EPS metal emissions

The original EPS inventory identifies two PM10-labelled rows as palladium and rhodium emissions. The [two rules](../schemas/mappings/bafu-2026-biosphere-eps-metals.json) select the exact source filename, full flow signature, empty CAS, and original amount. The genuine PM10 row stays unresolved.

The helper supplies the source amount as temporary canonical text for bw2io matching, rejects duplicate source matches, and removes both temporary fields afterwards. It follows historical ecoinvent UUIDs to the current **Palladium II** and **Rhodium III** names; this is catalog continuity, not measured speciation. All amounts and uncertainty remain unchanged, and original labels remain in audit metadata. See the [source evidence](../docs/bafu-2026-biosphere-eps-metals.md).

In [ ]:
biosphere_eps_metals_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-eps-metals.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-eps-metals.json",
)
biosphere_eps_metals_result

## Apply source-based plastics conversions

The [32 rules](../schemas/mappings/bafu-2026-biosphere-plastics-conversions.json) convert 28 coal resource quantities from MJ to kg using their original net calorific values (11.9 MJ/kg for brown coal; 26.3 MJ/kg for hard coal). Each rule selects one reviewed source file, and retains the original label, unit, calorific value and source flow UUID. One ethylene peat resource also converts MJ to kg using its source-defined 8.4 MJ/kg and the catalog’s biotic classification; the original in-ground category remains in metadata. Existing BAFU inventory adjustments remain in place.

Three approved EPS rules convert arsenic trioxide, lead dioxide and zinc oxide to contained As, Pb and Zn using their documented formulas. The original oxide labels and mass basis remain in metadata. Standard LCIA uses the existing metal-ion factors, losing oxide-form distinctions, including Pb(IV) in lead dioxide. All 32 conversions scale amounts and uncertainty together through bw2io. [Evidence and limitations](../docs/bafu-2026-biosphere-plastics-conversions.md).

In [ ]:
biosphere_plastics_conversions_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-plastics-conversions.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-plastics-conversions.json",
)
biosphere_plastics_conversions_result

## Restore source-defined chemical identities

The [three source-file rules](../schemas/mappings/bafu-2026-biosphere-plastics-chemicals.json) restore generic “Trichloroethane” labels to the original source’s 1,1,2 isomer in the EPVC, SPVC and vinyl-chloride inventories. Each original flow has matching CAS 79-00-5, a kilogram basis and the same generic-air compartment. The rules retain the original labels and leave amounts and uncertainty unchanged. [Evidence and rejected candidates](../docs/bafu-2026-biosphere-plastics-chemicals.md).

In [ ]:
biosphere_plastics_chemicals_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-plastics-chemicals.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-plastics-chemicals.json",
)
biosphere_plastics_chemicals_result

## Apply the remaining historical name correspondence

The [historical-name migration](../schemas/mappings/bafu-2026-biosphere-historical-names.json) links 82 urban-air “Benzene, dichloro-” exchanges through an explicit bw2io name correspondence and ecoinvent’s documented 3.9 rename to 1,2-dichlorobenzene. The installed 3.10 flow keeps the same UUID under its synonym “o-Dichlorobenzene”. Amounts, uncertainty, the urban-air compartment and original labels are preserved. This uses the historical catalog definition; it does not infer a composition for an unspecified isomer mixture. [Evidence](../docs/bafu-2026-biosphere-historical-names.md).

The same stage links one “1-Methyl-2-pyrrolidinone” emission to “N-methyl-2-pyrrolidone”: NIST confirms these are synonyms for CAS 872-50-4. The generic-air compartment, kilogram unit, original label, amount and uncertainty are preserved. The two rules resolve 83 exchange occurrences.

In [ ]:
biosphere_historical_names_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-historical-names.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-historical-names.json",
)
biosphere_historical_names_result

## Apply the approved NO₂ aggregation

The [two source-file rules](../schemas/mappings/bafu-2026-biosphere-nitrogen-dioxide.json) link the reviewed EPS and float-glass NO₂ emissions to `Nitrogen oxides / air / kilogram`, using the documented NOx-as-NO₂ mass basis. Amounts, uncertainty and original labels remain unchanged. This is the user-approved aggregate representation: standard LCIA loses separate NO₂ characterization, including the glass source’s deliberate NO/NO₂ distinction. [Evidence and decision](../docs/bafu-2026-biosphere-nitrogen-dioxide-proposal.md).

In [ ]:
biosphere_nitrogen_dioxide_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-nitrogen-dioxide.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-nitrogen-dioxide.json",
)
biosphere_nitrogen_dioxide_result

### Restore source-defined mosaic agricultural land

These 18 source-file-specific rules map mosaic agriculture to the equivalent heterogeneous agricultural class. Six transformation-to labels lost “mosaic” during export. Original labels, directions, quantities and uncertainty are retained. See [the source review](../docs/bafu-2026-biosphere-land-mosaic.md).

In [ ]:
biosphere_land_mosaic_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-land-mosaic.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-land-mosaic.json",
)
biosphere_land_mosaic_result

### Apply the approved fossil-carbon approximation

These 12 source-file-specific CO₂/CO rules use fossil targets for the reviewed diesel/demolition inventories. This is the approved fossil-default assumption despite a biogenic fuel share. Amounts, uncertainty and original labels are preserved. See [the carbon review](../docs/bafu-2026-biosphere-targeted-reassessment.md).

In [ ]:
biosphere_diesel_carbon_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-diesel-carbon.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-diesel-carbon.json",
)
biosphere_diesel_carbon_result

### Apply the approved uranium energy convention

Seven source-file rules convert MJ to uranium mass using the approved **560,000 MJ/kg** convention. This factor is an assumption, not a source-specific measurement; the alternative ILCD convention is 544,284 MJ/kg. Amounts and uncertainty scale together. Original labels, units and compartments plus the conversion assumption remain in exchange metadata. See [the evidence and decisions](../docs/bafu-2026-biosphere-targeted-reassessment.md#approved-uranium-and-wood-conversions).

In [ ]:
biosphere_uranium_convention_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-uranium-convention.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-uranium-convention.json",
)
biosphere_uranium_convention_result

### Apply the approved standing-wood density

Two source-file rules use the approved catalog default of **632.5 kg/m³** despite unknown source moisture. Paper wood uses `kg / 632.5`; EPS restores the source MJ basis and uses `MJ / 14.7 / 632.5`. Amounts and uncertainty scale together. Original labels, units and compartments plus the conversion assumption remain in exchange metadata. See [the evidence and decisions](../docs/bafu-2026-biosphere-targeted-reassessment.md#approved-uranium-and-wood-conversions).

In [ ]:
biosphere_wood_density_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-wood-density.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked-after-wood-density.json",
)
biosphere_wood_density_result

### Restore metal pairs mislabeled as PM10

Original PlasticsEurope inventories identify 13 pairs as palladium and rhodium. Their exported PM10 records have identical amounts and uncertainty. These 26 rules restore one of each metal per file using historical catalog correspondence, retaining all numerical fields and original labels. The helper requires exactly two identical source records and rejects missing, extra, different or partially migrated pairs. Genuine PM10 rows remain unresolved. [Source evidence and limits](../docs/bafu-2026-biosphere-pm10-metals.md).

In [ ]:
biosphere_pm10_metals_result = apply_biosphere_flow_migration(
    importer,
    ROOT / "schemas/mappings/bafu-2026-biosphere-pm10-metals.json",
    BIOSPHERE,
    ROOT / "reports/generated/biosphere-unlinked.json",
)
biosphere_pm10_metals_result

## Review the remaining biosphere flows

Create JSON and CSV worklists from this importer and the current biosphere catalog in `reports/generated/biosphere-remaining-worklist.*`. Candidates are review leads only: matching names, synonyms, or CAS numbers do not establish equivalent substances, units, or compartments. This audit does not change exchanges.

In [ ]:
from audit_unlinked_biosphere import audit_unlinked_biosphere

biosphere_audit_result = audit_unlinked_biosphere(importer, BIOSPHERE, ROOT)
biosphere_audit_result

In [ ]:
importer.statistics()

In [ ]:
importer.data[0]

In [ ]:
for u in list(importer.unlinked)[:10]:
    if u["type"] == "biosphere":
        print(u)

## Optional database write and LCA

Keep unsupported flows unresolved pending supported targets. The check below stops **Run All** before the existing drop/write/LCA cells while any exchange remains unlinked. The migration and diagnostic cells above retain all inventory rows. Do not run the later `drop_unlinked` cell while following this preservation workflow.

In [ ]:
unlinked_count = sum(1 for _ in importer.unlinked)
if unlinked_count:
    raise RuntimeError(
        f"{unlinked_count:,} exchanges remain unresolved. "
        "Review reports/generated/biosphere-unlinked.json and retain these exchanges; "
        "database writing is deferred until supported targets are established."
    )

In [ ]:
importer.drop_unlinked(i_am_reckless=True)

In [ ]:
importer.write_database()

In [ ]:
import bw2calc as bc
method = bd.methods.random()
act = bd.Database("BAFU:2026").random()
lca = bc.LCA({act: 1}, method)
lca.lci()
lca.lcia()
print(lca.score)

In [ ]:
act.as_dict()